# Text Mining — Don Quijote (Python)
Este notebook replica el análisis del script R `Text Mining - Don Quijote.R` en Python: carga de PDFs, limpieza, tokenización, conteos, wordcloud, bigramas y grafo de co-ocurrencia.

**Suposiciones:** Los archivos `DONQUIJOTE_PARTE1.pdf` y `DONQUIJOTE_PARTE2.pdf` están en la misma carpeta `task3/`.

**Estructura:**
1. Instalación / requisitos
2. Carga de PDFs
3. Limpieza y segmentación en frases
4. Tokenización y stopwords (es)
5. Conteos y visualizaciones
6. Bigramas y grafo

In [ ]:
# Requerimientos: ejecutar una vez para instalar paquetes (opcional)
# !pip install pdfplumber pandas nltk wordcloud matplotlib seaborn scikit-learn networkx pyvis tqdm

# Importar librerías
import os
import re
import pdfplumber
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx
from itertools import combinations
from tqdm import tqdm

nltk.download('punkt')
nltk.download('stopwords')

print('librerias cargadas')

In [ ]:
# 1. Leer PDFs (manejo de rutas y mensajes claros)
base_path = os.path.join(os.getcwd())
pdf1 = os.path.join(base_path, 'DONQUIJOTE_PARTE1.pdf')
pdf2 = os.path.join(base_path, 'DONQUIJOTE_PARTE2.pdf')

missing = [p for p in [pdf1, pdf2] if not os.path.exists(p)]
if missing:
    print('ATENCIÓN: No se encontraron los siguientes archivos PDF en', base_path)
    for m in missing:
        print(' -', os.path.basename(m))
    print('
    texto01 = []
    texto02 = []
else:
    def read_pdf(path):
        text_pages = []
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:






len(texto01), len(texto02)    texto02 = read_pdf(pdf2)    texto01 = read_pdf(pdf1)        return text_pages                text_pages.append(p.extract_text() or '')len(texto01), len(texto02)

In [ ]:
# 2. Unir y limpiar texto
texto = ' '.join(texto01 + texto02)
# Remover retornos de carro y saltos de linea
texto = texto.replace('
', ' ').replace('
', ' ')
# Remover enlaces comunes y repetidos de cabecera/pie
texto = re.sub(r'http[s]?://\S+', '', texto)
# Remplazar multiples espacios por uno
texto = re.sub(r'\s+', ' ', texto).strip()

# Segmentar en frases por punto
frases = [s.strip() for s in re.split(r'\. ', texto) if s.strip()]
df = pd.DataFrame({'frase': frases})
df.head()

In [ ]:
# 3. Limpieza manual similar al script R
removals = ['El Ingenioso Hidalgo Don Quijote de la Mancha','PRIMERA PARTE','Miguel de Cervantes Saavedra','Portal Educativo EducaCYL']
for r in removals:
    df['frase'] = df['frase'].str.replace(r, '', regex=False)
df['frase'] = df['frase'].str.strip()
df = df[df['frase'] != '']
df = df.reset_index(drop=True)
df.head()

In [ ]:
# 4. Tokenizacion y stopwords en español
sw = set(stopwords.words('spanish'))
# Añadir palabras adicionales del script R
sw.update(['capítulo','d'])

def tokenize_and_filter(text):
    tokens = [w.lower() for w in word_tokenize(text, language='spanish') if re.search('[a-zA-Zñáéíóúü]', w)]
    tokens = [t for t in tokens if t not in sw and len(t) > 1]
    return tokens

df['tokens'] = df['frase'].apply(tokenize_and_filter)
df.head()

In [ ]:
# 5. Conteo de palabras
all_tokens = [t for tokens in df['tokens'] for t in tokens]
counts = Counter(all_tokens)
top40 = counts.most_common(40)
pd.DataFrame(top40, columns=['word','count']).set_index('word')

In [ ]:
# Visualizar top40
top_df = pd.DataFrame(top40, columns=['word','count'])
plt.figure(figsize=(8,10))
sns.barplot(x='count', y='word', data=top_df, palette='Blues_d')
plt.title('Palabras más utilizadas (stopwords removidas)')
plt.show()

In [ ]:
# Wordcloud
wc = WordCloud(background_color='white', width=800, height=400).generate_from_frequencies(counts)
plt.figure(figsize=(16,8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
# 6. Bigramas por frase usando CountVectorizer (ngrams=2)
vectorizer = CountVectorizer(ngram_range=(2,2), stop_words=list(sw))
X = vectorizer.fit_transform(df['frase'])
bigrams = zip(vectorizer.get_feature_names_out(), X.toarray().sum(axis=0))
bigram_counts = sorted(bigrams, key=lambda x: x[1], reverse=True)
bigram_counts[:30]

In [ ]:
# 7. Grafo de co-ocurrencia: contar palabras por frase y crear aristas para pares que co-ocurren en la misma frase
min_count = 100
# construir parejas
pair_counts = Counter()
for tokens in df['tokens']:
    for a,b in combinations(set(tokens), 2):
        pair_counts[tuple(sorted((a,b)))] += 1

# Filtrar pares
edges = [(a,b,c) for (a,b),c in pair_counts.items() if c >= min_count]
len(edges)

In [ ]:
# Crear grafo y visualizar
G = nx.Graph()
for a,b,c in edges:
    G.add_edge(a,b,weight=c)

plt.figure(figsize=(12,12))
pos = nx.spring_layout(G, k=0.5)
weights = [G[u][v]['weight'] for u,v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=300, node_color='lightblue')
nx.draw_networkx_edges(G, pos, width=[w/5 for w in weights], edge_color='cyan')
nx.draw_networkx_labels(G, pos, font_size=10)
plt.title('Grafo de co-ocurrencia (pares con count >= {})'.format(min_count))
plt.axis('off')
plt.show()

## Notas finales
- Si los PDFs no están en `task3/`, moverlos allí o actualizar las rutas.
- Para un grafo interactivo, considerar `pyvis` o `plotly`.
- Este notebook asume codificación UTF-8 y usa NLTK para tokenizar y stopwords.